## Topic: Pipelines (Professional ML Workflow)
### Goal of Today

By the end of this session, we will:

- Understand ML pipelines
- Automate preprocessing + model
- Avoid common mistakes
- Write clean, reusable ML code

### 1. Problem Without Pipeline

👉 Until now you did:
```
Scaling → Train Model → Predict
```
But manually.

#### Problem:
```
Train data scaled  
Test data forgot scaling   
New data inconsistent
```
👉 Result:

- Wrong predictions
- Bugs in real apps

### 2. What is a Pipeline?

👉 Pipeline = Step-by-step automated workflow

Concept:
```
Input → Scaling → Model → Output
```
### Why important?
-  No manual mistakes
-  Clean code
-  Production ready

### Tools You Will Use
- Scikit-learn Pipeline
- StandardScaler
- LogisticRegression

Now we’ll turn this into a mini real workflow notebook and explain every line with comments so the concept becomes crystal clear.

We’ll reuse the dataset idea from previous lesson.

### Step 0 — Import Libraries

In [4]:
# Data split
from sklearn.model_selection import train_test_split

# Pipeline tools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

### Step 1 — Create Dataset

In [5]:
# Create fake classification dataset
X, y = make_classification(
    n_samples=1000,   # number of rows
    n_features=5,     # number of columns (features)
    random_state=42
)

# Split data into training & testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 20% for testing
    random_state=42
)

We now have:

- X_train → data used to learn
- X_test → unseen data to evaluate

### Step 2 — Create First Pipeline

In [7]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Step 1: scale features
    ('model', LogisticRegression()) # Step 2: train model
])

What this line means

Pipeline = automatic ML workflow

Inside list we define ordered steps:
| Step name | What it does    |
| --------- | --------------- |
| scaler    | normalize data  |
| model     | train algorithm |

Pipeline remembers the order and executes automatically.

### Step 3 — Train Pipeline

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression())])

What happens internally?

Pipeline secretly performs:
```
1) scaler.fit(X_train)        → learn mean & std
2) scaler.transform(X_train)  → scale data
3) model.fit(scaled_data)     → train model
```
All in ONE line

### Step 4 — Predict Using Pipeline

In [9]:
predictions = pipeline.predict(X_test)

Pipeline secretly performs
```
1) scaler.transform(X_test)   ← use SAME scaling learned from train
2) model.predict()
```
### Very important:
We never scale test data manually.

Pipeline prevents data leakage mistakes.

### Step 5 — Evaluate Model

In [10]:
print("Accuracy:", pipeline.score(X_test, y_test))

Accuracy: 0.885


score() automatically:

predicts
- compares with true labels
- returns accuracy

### BIG INSIGHT — Why Pipeline is Magic

Without pipeline we would need:
```
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
model.fit(X_train_scaled)

X_test_scaled = scaler.transform(X_test)
model.predict(X_test_scaled)
```

Many steps → easy to make mistakes

Pipeline = safe automation

### Step 6 — Predict New Customer (Real Use Case)

In [12]:
new_data = [[0.2, -1.3, 0.5, 1.2, -0.7]]

prediction = pipeline.predict(new_data)

print("Prediction:", prediction)

Prediction: [1]


What pipeline does automatically
```
New data → scaled → sent to model → prediction returned
```
This is exactly how real apps & APIs work.

### Step 7 — Pipeline + Cross Validation

In [13]:
scores = cross_val_score(
    pipeline,  # full pipeline
    X,         # full dataset
    y,
    cv=5       # 5-fold validation
)

print("CV Scores:", scores)
print("Average:", scores.mean())

CV Scores: [0.875 0.835 0.91  0.84  0.845]
Average: 0.861


What happens inside EACH fold?

For each split:
```
Fold 1:
  fit scaler on training fold
  train model
  test on validation fold

Fold 2:
  repeat again from scratch
```
Scaling is done inside each fold correctly.

This is why companies ALWAYS use pipeline + CV together.

### Step 8 — Switch Model Easily

We replace Logistic Regression with Decision Tree.

In [14]:
pipeline_tree = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=3))
])

👉 Only ONE line changed:
```
LogisticRegression → DecisionTreeClassifier
```
Everything else stays same

Train again:
```
pipeline_tree.fit(X_train, y_train)
print("Tree Accuracy:", pipeline_tree.score(X_test, y_test))
```

In [15]:
pipeline_tree.fit(X_train, y_train)
print("Tree Accuracy:", pipeline_tree.score(X_test, y_test))

Tree Accuracy: 0.865


### Step 9 — Real Industry Workflow

Real companies use pipelines like this:
```
Raw Data
   ↓
Cleaning
   ↓
Scaling
   ↓
Feature Engineering
   ↓
Model
   ↓
Prediction API
```

Pipeline = backbone of production ML systems.

### FINAL MASTER CONCEPT

Pipeline = ML assembly line

Instead of doing steps manually:

```
Step1 → Step2 → Step3 → Step4
```
Pipeline bundles everything into ONE reusable object.